# MIMIC-IV Hypotension: Grad-Attention Rollout + ViT Autoencoder Embeddings

This notebook trains or loads a ViT hypotension classifier, generates **only** gradient attention rollout explanations, trains or loads one ViT autoencoder on paired `[explanation, clinical value]` maps, creates latent embeddings for the selected split, and clusters those embeddings for value heatmaps.

## 1. Paths And Imports

In [ ]:
from pathlib import Path
import json
import os
import sys

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / 'src' / 'interpretable_ts_vit').exists():
        PROJECT_DIR = candidate
        break
else:
    raise FileNotFoundError(f'Could not find repository root from {cwd}')

SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.chdir(PROJECT_DIR)

RECORDS_PATH = PROJECT_DIR / 'data' / 'hypotension' / 'mimic_hypotension' / 'records.csv'
LABELS_PATH = PROJECT_DIR / 'data' / 'hypotension' / 'mimic_hypotension' / 'labels.csv'
if not RECORDS_PATH.exists() or not LABELS_PATH.exists():
    raise FileNotFoundError('Expected records.csv and labels.csv under data/hypotension/mimic_hypotension/.')

print('PROJECT_DIR:', PROJECT_DIR)
print('RECORDS_PATH:', RECORDS_PATH)
print('LABELS_PATH:', LABELS_PATH)

## 2. Runtime Check

In [ ]:
import numpy as np
import pandas as pd
import torch

import interpretable_ts_vit

print('interpretable_ts_vit:', interpretable_ts_vit.__file__)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Optional Balanced Subset

The default uses the full prepared MIMIC hypotension CSVs. Set `USE_SUBSET = True` for a quick smoke run.

In [ ]:
USE_SUBSET = False
PATIENTS_PER_CLASS = 120
RANDOM_STATE = 13

POC_DATA_DIR = PROJECT_DIR / 'data' / 'hypotension' / 'mimic_hypotension_local_poc'
POC_RECORDS_PATH = POC_DATA_DIR / 'records.csv'
POC_LABELS_PATH = POC_DATA_DIR / 'labels.csv'

if USE_SUBSET and not (POC_RECORDS_PATH.exists() and POC_LABELS_PATH.exists()):
    POC_DATA_DIR.mkdir(parents=True, exist_ok=True)
    labels = pd.read_csv(LABELS_PATH)
    per_class = min(PATIENTS_PER_CLASS, int(labels['label'].value_counts().min()))
    sampled_labels = (
        labels.groupby('label', group_keys=False)
        .sample(n=per_class, random_state=RANDOM_STATE)
        .sort_values('patient_id')
    )
    sampled_ids = set(sampled_labels['patient_id'].astype(str))
    sampled_labels.to_csv(POC_LABELS_PATH, index=False)

    first = True
    kept_records = 0
    for chunk in pd.read_csv(RECORDS_PATH, chunksize=500_000, dtype={'patient_id': str}):
        selected = chunk[chunk['patient_id'].isin(sampled_ids)]
        if len(selected):
            selected.to_csv(POC_RECORDS_PATH, index=False, mode='w' if first else 'a', header=first)
            first = False
            kept_records += len(selected)
    if first:
        pd.DataFrame(columns=['patient_id', 'variable', 'value', 'timestamp']).to_csv(POC_RECORDS_PATH, index=False)
    print(f'Created subset: {len(sampled_labels)} patients, {kept_records} records')

active_records_path = POC_RECORDS_PATH if USE_SUBSET else RECORDS_PATH
active_labels_path = POC_LABELS_PATH if USE_SUBSET else LABELS_PATH
print('Active records:', active_records_path)
print('Active labels:', active_labels_path)
display(pd.read_csv(active_labels_path)['label'].value_counts().rename('patients'))

## 4. Configuration

The explanation method is fixed to `grad_attention_rollout`. Autoencoder work uses a ViT-style patch encoder/decoder and is separated into encoder training, embedding creation, and embedding clustering.

In [ ]:
import shutil

from interpretable_ts_vit.autoencoder import (
    cluster_autoencoder_embeddings,
    create_explanation_value_embeddings,
    train_explanation_value_autoencoder,
)
from interpretable_ts_vit.binning import TimeSeriesBinner
from interpretable_ts_vit.config import DataConfig, ModelConfig, TrainConfig, ExplainConfig, ClusterConfig
from interpretable_ts_vit.data_modules import MIMICHypotensionDataModule
from interpretable_ts_vit.io import load_model, load_split
from interpretable_ts_vit.model_modules import ViTTimeSeriesModule
from interpretable_ts_vit.pipeline import _denormalized_patient_value_maps

RUN_DIR = PROJECT_DIR / 'runs' / 'full_hypotension_importance_values'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'hypotension' / 'processed_full_hypotension_importance_values'
SPLIT = 'test'

RESET_OUTPUTS = False
LOAD_SAVED_ARTIFACTS = True

if RESET_OUTPUTS:
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    shutil.rmtree(PROCESSED_DIR, ignore_errors=True)

IMPORTANCE_PLOT_MODE = 'value_with_importance_opacity'
IMPORTANCE_THRESHOLD = 0.8
SHOW_MEAN_VALUES = False
USE_NORMAL_RANGES = True

EXPLANATION_BATCH_SIZE = 16

AUTOENCODER_LATENT_DIM = 16
AUTOENCODER_EPOCHS = 50
AUTOENCODER_BATCH_SIZE = 64
AUTOENCODER_LEARNING_RATE = 1e-3
AUTOENCODER_PATIENCE = 10


data_config = DataConfig(granularity='0.5h', aggregation='mean', val_fraction=0.2, test_fraction=0.2, random_state=RANDOM_STATE)
model_config = ModelConfig(patch_size=(1, 4), embed_dim=64, depth=2, num_heads=4, mlp_ratio=2.0, dropout=0.1)
train_config = TrainConfig(
    batch_size=64,
    epochs=50,
    learning_rate=1e-3,
    weight_decay=1e-4,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    early_stopping_patience=8,
    early_stopping_monitor='val_loss',
    restore_best_model=True,
    verbose=True,
    progress_interval_batches=25,
)
explain_config = ExplainConfig(method='grad_attention_rollout', target_class=1, batch_size=EXPLANATION_BATCH_SIZE)
cluster_config = ClusterConfig(
    method='kmeans',
    n_clusters=4,
    feature_mode='autoencoder',
    autoencoder_latent_dim=AUTOENCODER_LATENT_DIM,
    autoencoder_epochs=AUTOENCODER_EPOCHS,
    autoencoder_learning_rate=AUTOENCODER_LEARNING_RATE,
    autoencoder_batch_size=AUTOENCODER_BATCH_SIZE,
    autoencoder_early_stopping_patience=AUTOENCODER_PATIENCE,
    plot_mode=IMPORTANCE_PLOT_MODE,
    importance_threshold=IMPORTANCE_THRESHOLD,
    show_values=SHOW_MEAN_VALUES,
    use_normal_ranges=USE_NORMAL_RANGES,
)

data = MIMICHypotensionDataModule(
    records_path=active_records_path,
    labels_path=active_labels_path,
    processed_dir=PROCESSED_DIR,
    data_config=data_config,
)
model = ViTTimeSeriesModule(RUN_DIR, model_config, train_config, explain_config, cluster_config)
print('processed:', PROCESSED_DIR)
print('run:', RUN_DIR)
print('split:', SPLIT)

## 5. Prepared Tensor Data

This cell makes it explicit which prepared tensors are used downstream. If the files exist, they are loaded instead of recomputed.

In [ ]:
required_prepared = [PROCESSED_DIR / 'binner.json', *(PROCESSED_DIR / f'{name}.npz' for name in ['train', 'val', 'test'])]
if LOAD_SAVED_ARTIFACTS and all(path.exists() for path in required_prepared):
    print('Loaded cached prepared tensors from:', PROCESSED_DIR)
else:
    data.prepare()
    print('Prepared and saved tensors to:', PROCESSED_DIR)

data.load()
for split_name in ['train', 'val', 'test']:
    split_ds = data.split(split_name)
    print(f'{split_name}: {len(split_ds)} patients | tensor shape {tuple(split_ds.x.shape)} | file {data.split_path(split_name)}')
print('variables:', data.variable_vocab)
print('labels:', data.label_names)

## 6. Classifier Weights

This cell loads `model.pt` when available; otherwise it trains the ViT and saves weights plus training metrics.

In [ ]:
model_path = RUN_DIR / 'model.pt'
train_metrics_path = RUN_DIR / 'metrics.json'
if LOAD_SAVED_ARTIFACTS and model_path.exists():
    loaded_model = load_model(RUN_DIR)
    train_metrics = json.loads(train_metrics_path.read_text()) if train_metrics_path.exists() else None
    print('Loaded saved model weights:', model_path)
else:
    train_metrics = model.fit(data)
    print('Trained and saved model weights:', model_path)

if train_metrics:
    print('best epoch:', train_metrics.get('best_epoch'))
    print('epochs ran:', train_metrics.get('epochs_ran'))
    print('best monitor value:', train_metrics.get('best_monitor_value'))

## 7. Evaluation

In [ ]:
metrics_path = model.metrics_path(SPLIT)
predictions_path = model.predictions_path(SPLIT)
if LOAD_SAVED_ARTIFACTS and metrics_path.exists() and predictions_path.exists():
    evaluation_metrics = json.loads(metrics_path.read_text())
    print('Loaded cached evaluation:', metrics_path)
else:
    evaluation_metrics = model.evaluate(data, split=SPLIT)
    print('Evaluated and saved:', metrics_path)

print(json.dumps(evaluation_metrics, indent=2))
display(pd.read_csv(predictions_path).head())

## 8. Gradient Attention Rollout Explanations

Explanations are cached as one `.npy` map per patient. The autoencoder needs train, validation, and target split explanations.

In [ ]:
def ensure_explanations(split_name):
    out_dir = model.explanations_dir(split_name)
    model.explain(data, split=split_name, show_progress=True, batch_size=EXPLANATION_BATCH_SIZE)
    print(f'{split_name}: gradient attention rollout explanations are ready in {out_dir}')
    return out_dir

explanation_dirs = {split_name: ensure_explanations(split_name) for split_name in ['train', 'val', SPLIT]}

## 9. Autoencoder Data

Load the paired explanation and denormalized clinical-value maps used by the next three stages.

In [ ]:
binner = TimeSeriesBinner.load(RUN_DIR / 'binner.json')
train_dataset = load_split(RUN_DIR / 'train.npz')
val_dataset = load_split(RUN_DIR / 'val.npz')
test_dataset = load_split(RUN_DIR / f'{SPLIT}.npz')

train_values = _denormalized_patient_value_maps(train_dataset, binner)
val_values = _denormalized_patient_value_maps(val_dataset, binner)
test_values = _denormalized_patient_value_maps(test_dataset, binner)
clusters_dir = model.clusters_dir(SPLIT)

print('Autoencoder data:')
print('  train explanations:', explanation_dirs['train'])
print('  train values:', len(train_values), 'patients from', RUN_DIR / 'train.npz')
print('  validation explanations:', explanation_dirs['val'])
print('  validation values:', len(val_values), 'patients from', RUN_DIR / 'val.npz')
print('  target explanations:', explanation_dirs[SPLIT])
print('  target values:', len(test_values), 'patients from', RUN_DIR / f'{SPLIT}.npz')
print('  artifacts:', clusters_dir)


## 10. Train ViT Encoder

Train or load the ViT autoencoder on train `[explanation, value]` maps and validate it on validation maps. Progress is printed once per epoch when training is not loaded from cache.

In [ ]:
trained_encoder = train_explanation_value_autoencoder(
    explanation_dirs['train'],
    train_values,
    validation_explanations=explanation_dirs['val'],
    validation_values=val_values,
    output_dir=clusters_dir,
    latent_dim=AUTOENCODER_LATENT_DIM,
    epochs=AUTOENCODER_EPOCHS,
    learning_rate=AUTOENCODER_LEARNING_RATE,
    batch_size=AUTOENCODER_BATCH_SIZE,
    device=train_config.device,
    early_stopping_patience=AUTOENCODER_PATIENCE,
    show_progress=True,
)

print('ViT autoencoder artifact:', clusters_dir / 'autoencoder.pt')
print('Loaded from cache:', trained_encoder.get('loaded_from_cache', False))
print(json.dumps({k: v for k, v in trained_encoder['metrics'].items() if k != 'history'}, indent=2))


## 11. Create Test Embeddings

Use the trained ViT encoder to create latent embeddings for the selected target split and report reconstruction metrics for that split.

In [ ]:
embedded_split = create_explanation_value_embeddings(
    explanation_dirs[SPLIT],
    test_values,
    model=trained_encoder['model'],
    preprocessor=trained_encoder['preprocessor'],
    output_dir=clusters_dir,
    batch_size=AUTOENCODER_BATCH_SIZE,
    device=train_config.device,
)

print('Embedding artifact:', clusters_dir / 'autoencoder_embeddings.csv')
print('Loaded from cache:', embedded_split.get('loaded_from_cache', False))
print('Embeddings shape:', embedded_split['embeddings'].shape)
test_metrics = {
    'train_loss': trained_encoder['metrics']['train_loss'],
    'val_loss': trained_encoder['metrics']['val_loss'],
    f'{SPLIT}_reconstruction_loss': embedded_split['loss'],
    'best_epoch': trained_encoder['metrics']['best_epoch'],
    'n_test_patients': len(embedded_split['patient_ids']),
}
print('Test metrics:')
display(pd.DataFrame([test_metrics]))
display(embedded_split['embedding_frame'].head())


## 12. Cluster Test Embeddings

Cluster the saved ViT autoencoder embedding table and write cluster assignments, centroids, and aggregate explanation maps.

In [ ]:
autoencoder_metrics = trained_encoder['metrics'] | {'cluster_loss': embedded_split['loss']}
autoencoder_metadata = {
    **trained_encoder['metadata'],
    'cluster': {'n_patients': len(embedded_split['patient_ids']), **embedded_split['metadata']},
}

clustered = cluster_autoencoder_embeddings(
    embedded_split['embedding_frame'],
    explanations=embedded_split['explanations'],
    predictions=model.predictions_path(SPLIT),
    n_clusters=cluster_config.n_clusters,
    method=cluster_config.method,
    output_dir=clusters_dir,
    autoencoder_metrics=autoencoder_metrics,
    autoencoder_metadata=autoencoder_metadata,
    hdbscan_min_cluster_size=cluster_config.hdbscan_min_cluster_size,
    hdbscan_min_samples=cluster_config.hdbscan_min_samples,
)

print('Cluster artifacts:', clusters_dir)
print('Assignments:', clusters_dir / 'cluster_assignments.csv')
display(clustered['assignments'].head())


## 13. Cluster Value Heatmaps

In [ ]:
from IPython.display import Image, display

heatmaps_dir = model.plot_cluster_values(data, split=SPLIT)
heatmaps = sorted(heatmaps_dir.rglob('*.png'))
print('Cluster heatmaps:', heatmaps_dir)
print('Found:', len(heatmaps))
for path in heatmaps:
    print(path.relative_to(heatmaps_dir))
    display(Image(filename=str(path)))

## 14. Saved Artifacts

In [ ]:
important_artifacts = [
    PROCESSED_DIR / 'train.npz',
    PROCESSED_DIR / 'val.npz',
    PROCESSED_DIR / 'test.npz',
    RUN_DIR / 'model.pt',
    RUN_DIR / 'metrics.json',
    model.metrics_path(SPLIT),
    model.predictions_path(SPLIT),
    model.clusters_dir(SPLIT) / 'autoencoder.pt',
    model.clusters_dir(SPLIT) / 'autoencoder_embeddings.csv',
    model.clusters_dir(SPLIT) / 'autoencoder_metrics.json',
    model.clusters_dir(SPLIT) / 'autoencoder_training_metadata.json',
    model.clusters_dir(SPLIT) / 'autoencoder_embedding_metadata.json',
    model.clusters_dir(SPLIT) / 'cluster_assignments.csv',
]
for path in important_artifacts:
    print(('OK ' if path.exists() else 'MISSING '), path)